# Keras 核心 API 全流程实战

**学习路线：**

| 阶段 | 章节 | 内容 | 关键词 |
|------|------|------|--------|
| **一、模型构建** | §1-§4 | 层/模型/API 基础 | Layer, Model, Sequential, 函数式 |
| **进阶示例** | §5 | Transformer Encoder | 自定义层高级用法（可先跳过） |
| **二、训练流程** | §6-§9 | 编译/训练/评估/可视化 | compile, fit, evaluate, predict |
| **三、工程实践** | §10-§11 | 保存/回调 | save, Callbacks |

> **学习建议**：§5 为进阶内容，展示复杂自定义层的写法，可先跳过，学完训练流程后再回看。
>
> 全部使用合成数据，可直接运行。跑通后替换成自己的数据即可。

In [ ]:
# === 基础导入 ===
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow: {tf.__version__}")
print(f"Keras: {keras.__version__}")

---
# 第一阶段：模型构建

本阶段学习 Keras 的核心概念：**层（Layer）** 和 **模型（Model）**，
掌握两种搭建模型的方式：Sequential 和函数式 API。

---
## §1 层与模型的关系

**层（Layer）** = 数据处理模块，输入张量 → 输出张量，内部可能有可训练参数。

**模型（Model）** = 多个层组合在一起的结构，Model 本身也是 Layer 的子类。

| 概念 | 封装的内容 |
|------|----------|
| Layer | **状态**（权重 W、偏置 b）+ **计算**（前向传播逻辑） |
| Model | 多个 Layer 的拓扑连接 + `compile`/`fit`/`evaluate`/`predict` 方法 |

---
## §2 自定义 Dense 层

自定义层需要实现两个核心方法：

| 方法 | 调用时机 | 职责 |
|------|---------|------|
| `build(input_shape)` | 第一次收到输入时 | 根据输入维度创建权重 |
| `call(inputs)` | 每次前向传播 | 定义计算逻辑 |

**Dense 层的数学本质：** `output = activation(X @ W + b)`

In [ ]:
class SimpleDense(keras.layers.Layer):
    """自定义全连接层：output = activation(X @ W + b)"""
    
    def __init__(self, units, activation=None, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.activation = activation

    def build(self, input_shape):
        # 第一次调用时才创建权重，实现"自动形状推断"
        input_dim = input_shape[-1]
        self.W = self.add_weight(
            shape=(input_dim, self.units),
            initializer="random_normal",
            trainable=True, name="kernel"
        )
        self.b = self.add_weight(
            shape=(self.units,),
            initializer="zeros",
            trainable=True, name="bias"
        )

    def call(self, inputs):
        y = tf.matmul(inputs, self.W) + self.b
        if self.activation is not None:
            y = self.activation(y)
        return y

In [ ]:
# 测试自定义层
my_dense = SimpleDense(units=32, activation=tf.nn.relu)
input_tensor = tf.ones(shape=(2, 784))
output_tensor = my_dense(input_tensor)
print(f"输入: {input_tensor.shape}  →  输出: {output_tensor.shape}")

# 查看参数
print(f"权重 W 形状: {my_dense.W.shape}")
print(f"偏置 b 形状: {my_dense.b.shape}")
print(f"总参数量: {784 * 32 + 32} = {784 * 32} (权重) + {32} (偏置)")

---
## §3 自动形状推断

### 为什么 Keras 不用写 `input_dim`？

**传统写法（笨）：** 必须在初始化时指定输入维度
```python
layer = Dense(input_dim=784, output_dim=32)  # 必须写 input_dim
```

**Keras 写法（聪明）：** 第一次调用时自动推断
```python
layer = Dense(32)        # 只指定输出维度
output = layer(input)    # 第一次调用时自动创建权重
```

下面用一个对比示例展示两种方式的区别：

In [ ]:
# "笨"版本：必须在初始化时指定 input_size 和 output_size
class NaiveDense:
    """不够灵活的全连接层实现"""
    def __init__(self, input_size, output_size, activation=None):
        self.activation = activation
        self.W = tf.Variable(tf.random.normal((input_size, output_size)))
        self.b = tf.Variable(tf.zeros((output_size,)))
    
    def __call__(self, inputs):
        y = tf.matmul(inputs, self.W) + self.b
        if self.activation:
            y = self.activation(y)
        return y

In [ ]:
# 用笨版本搭建多层网络：每一层的 input_size 都要手写，而且必须对上！
naive_layers = [
    NaiveDense(784, 32, tf.nn.relu),   # input_size=784, output_size=32
    NaiveDense(32, 64, tf.nn.relu),    # input_size=32 (必须等于上一层的 output_size!)
    NaiveDense(64, 10, tf.nn.softmax)  # input_size=64 (必须等于上一层的 output_size!)
]

x = tf.ones((2, 784))
for layer in naive_layers:
    x = layer(x)
print(f"输出: {x.shape}")

print("\n问题：如果我想把第二层改成 output_size=128，")
print("那第三层的 input_size 也要跟着改成 128...")
print("改一个地方，动一堆代码，容易出错！")

### 参数总量计算

全连接层参数数量公式：

$$\text{参数量} = \text{输入维度} \times \text{输出维度} + \text{输出维度}$$

即：**权重参数 + 偏置参数**

例如：输入 784 维，输出 32 维
- 权重：784 × 32 = 25,088
- 偏置：32
- 总计：25,120 个参数

In [ ]:
# 用 Keras 风格（自动形状推断）搭建同样的模型
model_custom = keras.Sequential([
    SimpleDense(32, activation=tf.nn.relu),   # 不用写 input_dim！
    SimpleDense(64, activation=tf.nn.relu),   # 自动知道输入是 32
    SimpleDense(10, activation=tf.nn.softmax) # 自动知道输入是 64
])

y = model_custom(tf.ones((4, 784)))  # 第一次调用时自动构建所有层
print(f"输出: {y.shape}")
model_custom.summary()

---
## §4 Sequential vs 函数式 API

### 两种搭建模型的方式

| 方式 | 适用场景 | 特点 |
|------|----------|------|
| Sequential | 简单的线性堆叠 | 代码简洁，一行搞定 |
| 函数式 API | 复杂结构（残差、多输入输出） | 灵活，可构建任意拓扑 |

**Sequential 的局限**：只能处理"一条线"的结构
```
Input → Layer1 → Layer2 → Layer3 → Output
```

**函数式 API**：可以处理复杂结构
```
     ┌──────────────┐
     │    Input     │
     └──────┬───────┘
            │
     ┌──────┴───────┐
     │              │
     ▼              ▼
  Layer1         (跳过)
     │              │
     └──────┬───────┘
            ▼
        Add (残差连接)
```

下面用两种方式搭建同样的 MLP（多层感知机）：

In [ ]:
# ========== 方式一：Sequential API ==========
sequential_model = keras.Sequential([
    layers.Dense(64, activation="relu", name="hidden_1"),
    layers.Dense(64, activation="relu", name="hidden_2"),
    layers.Dense(10, activation="softmax", name="output")
], name="sequential_mlp")

# 第一次调用才构建
sequential_model(tf.ones((2, 784)))
print("=== Sequential API ===")
sequential_model.summary()

In [ ]:
# ========== 方式二：函数式 API ==========
# 核心思想：每一层像函数一样调用，传入上一层的输出

# 1. 定义输入节点
inputs = keras.Input(shape=(784,), name="input_features")

# 2. 定义层和连接（注意语法：Layer()(上一层的输出)）
x = layers.Dense(64, activation="relu", name="hidden_1")(inputs)
x = layers.Dense(64, activation="relu", name="hidden_2")(x)
outputs = layers.Dense(10, activation="softmax", name="output")(x)

# 3. 创建模型（把从 inputs 到 outputs 的计算图打包）
functional_model = keras.Model(inputs=inputs, outputs=outputs, name="functional_mlp")

print("\n=== 函数式 API ===")
functional_model.summary()

### 第一阶段小结

| 概念 | 要点 |
|------|------|
| Layer | 封装状态（权重）和计算（前向传播） |
| `build()` | 第一次调用时创建权重，实现自动形状推断 |
| `call()` | 定义前向传播逻辑 |
| Sequential | 简单模型，层按顺序堆叠 |
| 函数式 API | 复杂模型，支持残差连接、多输入输出 |

---
# 进阶示例：复杂自定义层

> **说明**：本节展示如何用函数式 API 构建复杂的自定义组件。
> 这是一个 **进阶示例**，初学者可以 **先跳过**，学完第二阶段的训练流程后再回来理解。

---
## §5 高级示例：Transformer Encoder Block

Transformer 是现代大模型（GPT、BERT 等）的核心结构。

### 本节目的

通过实现一个 Transformer Encoder Block，学习：
1. 如何组合多个层构建复杂组件
2. 残差连接的实现方式
3. 层归一化（LayerNormalization）的作用

### Transformer Encoder 结构

```
Input
  │
  ├──────────────────────┐
  ▼                      │
MultiHeadAttention       │
  │                      │
  ▼                      │
Add ◄────────────────────┘  ← 残差连接：input + attention_output
  │
  ▼
LayerNorm
  │
  ├──────────────────────┐
  ▼                      │
Dense(relu) → Dense      │
  │                      │
  ▼                      │
Add ◄────────────────────┘  ← 残差连接：input + ffn_output
  │
  ▼
LayerNorm
  │
Output
```

### 关键概念

| 组件 | 作用 |
|------|------|
| MultiHeadAttention | 让序列中每个位置"看到"其他所有位置 |
| 残差连接 `x + layer(x)` | 缓解梯度消失，让深层网络可训练 |
| LayerNormalization | 稳定训练过程 |

In [ ]:
class TransformerEncoder(layers.Layer):
    """
    Transformer Encoder Block。
    
    结构：Input → MultiHeadAttention → Add&Norm → FeedForward → Add&Norm → Output
    
    参数
    ----
    embed_dim : int
        嵌入维度（输入/输出的特征维度）
    dense_dim : int
        前馈网络中间层维度（通常比 embed_dim 大）
    num_heads : int
        注意力头数量（embed_dim 必须能被 num_heads 整除）
    """
    
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        
        # 多头自注意力层
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads, 
            key_dim=embed_dim
        )
        
        # 前馈网络：先升维，再降回原尺寸
        self.dense_proj = keras.Sequential([
            layers.Dense(dense_dim, activation="relu"),  # 升维
            layers.Dense(embed_dim)                        # 降回
        ])
        
        # 两个层归一化
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
    
    def call(self, inputs):
        # 1. 自注意力（Q、K、V 都是 inputs 自己）
        attn_output = self.attention(inputs, inputs)
        
        # 2. 第一次残差连接 + 层归一化
        x = self.layernorm_1(inputs + attn_output)
        
        # 3. 前馈网络
        ffn_output = self.dense_proj(x)
        
        # 4. 第二次残差连接 + 层归一化
        return self.layernorm_2(x + ffn_output)

In [ ]:
# 测试 TransformerEncoder
# 输入：batch_size=2，序列长度=8，每个 token 的维度=32
dummy_seq = tf.random.normal((2, 8, 32))

# 实例化编码器
encoder = TransformerEncoder(embed_dim=32, dense_dim=64, num_heads=4)

# 前向传播
output = encoder(dummy_seq)

print(f"输入: {dummy_seq.shape} → 输出: {output.shape}")
print("\nTransformer 的特点：输入输出形状相同")
print("  - 序列长度不变（8 → 8）")
print("  - 特征维度不变（32 → 32）")
print("  - 但每个位置的表示融合了其他位置的信息")

---
# 第二阶段：训练流程

本阶段学习 Keras 的训练四步曲：**compile → fit → evaluate → predict**。

```
┌─────────────────────────────────────────────────────────────┐
│  Step 1: compile(optimizer, loss, metrics)  — 配置训练规则  │
│  Step 2: fit(x, y, epochs, batch_size)      — 训练模型      │
│  Step 3: evaluate(x, y)                      — 评估性能      │
│  Step 4: predict(x)                          — 预测新数据    │
└─────────────────────────────────────────────────────────────┘
```

---
## §6 模型编译与训练

### 训练流程四步

| 步骤 | 方法 | 作用 |
|------|------|------|
| 1 | `compile()` | 配置优化器、损失函数、监控指标 |
| 2 | `fit()` | 训练模型，更新参数 |
| 3 | `evaluate()` | 在测试集上评估性能 |
| 4 | `predict()` | 对新数据做预测 |

### 6.1 准备数据

先用合成数据演示完整流程：

In [ ]:
# 准备二分类数据
np.random.seed(42)
tf.random.set_seed(42)

# 生成模拟数据：2000个样本，每个20维特征
N, D = 2000, 20
inputs = np.random.randn(N, D).astype('float32')

# 随机构造分类边界
true_w = np.random.randn(D, 1).astype('float32')
logits = inputs @ true_w + 0.3 * np.random.randn(N, 1).astype('float32')

# 生成二分类标签（0 或 1）
targets = (logits > 0).astype('float32')

print(f"输入数据: {inputs.shape}")
print(f"标签数据: {targets.shape}")
print(f"类别分布 - 0: {(targets==0).sum()}, 1: {(targets==1).sum()}")

### 6.2 Step 1：compile() — 配置训练规则

compile() 的三大核心参数：

| 参数 | 作用 | 常见选择 |
|------|------|----------|
| `optimizer` | 决定参数怎么更新 | SGD, Adam, RMSprop |
| `loss` | 衡量预测与真实标签的差异 | MSE, CrossEntropy |
| `metrics` | 训练时监控的指标 | Accuracy, MAE |

In [ ]:
# 构建二分类模型
binary_model = keras.Sequential([
    layers.Dense(32, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')   # 输出 0~1 的概率
], name="binary_classifier")

# compile：配置优化器、损失函数、监控指标
binary_model.compile(
    optimizer='rmsprop',          # 优化器
    loss='binary_crossentropy',   # 损失函数（二分类）
    metrics=['accuracy']          # 监控指标
)

binary_model.summary()

### 6.3 Step 2：fit() — 训练模型

fit() 的核心参数：

| 参数 | 作用 |
|------|------|
| `x, y` | 输入数据和标签 |
| `epochs` | 训练轮数（遍历全部数据多少次） |
| `batch_size` | 每次更新用多少样本 |
| `validation_data` | 验证集（可选） |
| `verbose` | 打印信息的详细程度 |

In [ ]:
# 基础训练：无验证集
history = binary_model.fit(
    inputs, targets,
    epochs=5,         # 训练 5 轮
    batch_size=128,   # 每批 128 个样本
    verbose=1
)

# 2000 个样本 / 每批 128 = 每轮约 16 次更新

---
## §7 验证集与数据管道

### 7.1 为什么需要验证集？

| 数据集 | 作用 |
|--------|------|
| 训练集 | 用来更新参数 |
| 验证集 | 检查模型是否"记住"了数据（过拟合） |
| 测试集 | 最终评估模型性能 |

**过拟合现象**：训练集准确率很高，但验证集准确率很低 → 模型"死记硬背"了训练数据

In [ ]:
# 划分训练集和验证集
indices = np.random.permutation(N)
val_n = 400  # 20% 作为验证集

val_x, val_y = inputs[indices[:val_n]], targets[indices[:val_n]]
train_x, train_y = inputs[indices[val_n:]], targets[indices[val_n:]]

print(f"训练集: {train_x.shape[0]} 个样本")
print(f"验证集: {val_x.shape[0]} 个样本")

In [ ]:
# 带验证集的训练
model_v = keras.Sequential([
    layers.Dense(32, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
], name="binary_classifier_with_val")

model_v.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])

# validation_data：每轮结束后在验证集上评估
history_v = model_v.fit(
    train_x, train_y,
    epochs=10,
    batch_size=16,
    validation_data=(val_x, val_y),  # 验证集
    verbose=1
)

# history_v.history 会包含：loss, accuracy, val_loss, val_accuracy

### 7.2 使用 tf.data 构建高效数据管道

tf.data 的优势：
- 支持大数据集（不用一次性加载到内存）
- 自动批处理、打乱、预取
- 更高效的 GPU 利用

In [ ]:
# 用 tf.data 构建数据管道
train_ds = (tf.data.Dataset.from_tensor_slices((train_x, train_y))
            .shuffle(1024)               # 打乱
            .batch(32)                   # 分批
            .prefetch(tf.data.AUTOTUNE)) # 预取加速

val_ds = (tf.data.Dataset.from_tensor_slices((val_x, val_y))
          .batch(32)
          .prefetch(tf.data.AUTOTUNE))

# 用 Dataset 训练
model_ds = keras.Sequential([
    layers.Dense(32, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])
model_ds.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model_ds.fit(train_ds, epochs=3, validation_data=val_ds, verbose=1)

---
## §8 评估与预测

### 8.1 Step 3：evaluate() — 评估模型

在独立测试集上评估最终效果，不更新参数。

In [ ]:
# 在验证集上评估
results = model_v.evaluate(val_x, val_y, verbose=0)
print(f"evaluate() 返回: {results}")
print(f"  - loss: {results[0]:.4f}")
print(f"  - accuracy: {results[1]:.4f}")

### 8.2 Step 4：predict() — 预测新数据

对新的数据进行预测。

| 方式 | 语法 | 返回类型 | 适用场景 |
|------|------|----------|----------|
| 直接调用 | `model(x)` | tf.Tensor | 训练时、需要梯度 |
| predict() | `model.predict(x)` | NumPy 数组 | 批量推理、部署 |

In [ ]:
# 方式一：直接调用（返回 TensorFlow Tensor）
pred_tensor = model_v(val_x[:10])
print(f"model(x) 返回类型: {type(pred_tensor)}")

# 方式二：predict()（返回 NumPy 数组）
pred_np = model_v.predict(val_x[:10], verbose=0)
print(f"model.predict() 返回类型: {type(pred_np)}")

# 查看预测结果
print("\n预测结果示例：")
print(f"{'概率':>8} → {'预测类别':>6} | {'真实标签':>6}")
print("-" * 35)
for i, (p, t) in enumerate(zip(pred_np.flatten(), val_y[:10].flatten())):
    pred_class = int(p > 0.5)
    match = "✓" if pred_class == int(t) else "✗"
    print(f"{p:>8.4f} → {pred_class:>6} | {int(t):>6} {match}")

---
## §9 损失函数与指标

### 不同任务的损失函数选择

| 任务类型 | 标签格式 | 损失函数 | 输出层激活 |
|----------|----------|----------|------------|
| 二分类 | 0/1 整数 | `BinaryCrossentropy` | sigmoid |
| 多分类 | 整数 (0,1,2,...) | `SparseCategoricalCrossentropy` | softmax |
| 多分类 | one-hot 编码 | `CategoricalCrossentropy` | softmax |
| 回归 | 连续值 | `MeanSquaredError` | 无 |

In [ ]:
# 准备多分类数据
np.random.seed(123)

N_mc, D_mc, C = 1500, 16, 3  # 1500样本，16维特征，3个类别
x_mc = np.random.randn(N_mc, D_mc).astype('float32')

# 两种标签格式
y_int = np.random.randint(0, C, N_mc).astype('int32')           # 整数标签
y_onehot = keras.utils.to_categorical(y_int, C).astype('float32')  # one-hot标签

print(f"整数标签示例: {y_int[:5]}")
print(f"one-hot标签示例:")
for i in range(5):
    print(f"  类别 {y_int[i]} → {y_onehot[i]}")

In [ ]:
# ========== 多分类方式一：SparseCategoricalCrossentropy ==========
# 标签是整数，不需要手动转 one-hot

sparse_model = keras.Sequential([
    layers.Dense(32, activation='relu'),
    layers.Dense(C, activation='softmax')  # 输出 3 个概率
])

sparse_model.compile(
    optimizer='adam',
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy()]
)

print("=== SparseCategoricalCrossentropy ===")
print("标签格式：整数 [0, 1, 2, ...]")
sparse_model.fit(x_mc, y_int, epochs=3, batch_size=32, verbose=1)

In [ ]:
# ========== 多分类方式二：CategoricalCrossentropy ==========
# 标签是 one-hot 编码

cat_model = keras.Sequential([
    layers.Dense(32, activation='relu'),
    layers.Dense(C, activation='softmax')
])

cat_model.compile(
    optimizer='adam',
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=[keras.metrics.CategoricalAccuracy()]
)

print("\n=== CategoricalCrossentropy ===")
print("标签格式：one-hot [[1,0,0], [0,1,0], [0,0,1]]")
cat_model.fit(x_mc, y_onehot, epochs=3, batch_size=32, verbose=1)

In [ ]:
# ========== 回归任务：MeanSquaredError ==========

np.random.seed(7)
x_reg = np.random.randn(1200, 10).astype('float32')
y_reg = np.random.randn(1200, 1).astype('float32')

reg_model = keras.Sequential([
    layers.Dense(32, activation='relu'),
    layers.Dense(1)  # 注意：无激活函数，输出任意实数
])

reg_model.compile(
    optimizer='rmsprop',
    loss='mse',      # 均方误差
    metrics=['mae']  # 平均绝对误差
)

print("=== 回归任务 ===")
reg_model.fit(x_reg, y_reg, epochs=5, batch_size=32, verbose=1)

---
## §10 可视化训练过程

通过绘制 loss 和 accuracy 曲线，观察训练情况。

In [ ]:
import matplotlib.pyplot as plt

def plot_history(history):
    """绘制训练曲线"""
    h = history.history
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # 左图：损失曲线
    axes[0].plot(h['loss'], label='train_loss')
    if 'val_loss' in h:
        axes[0].plot(h['val_loss'], label='val_loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].set_title('损失曲线')
    
    # 右图：准确率曲线
    axes[1].plot(h['accuracy'], label='train_acc')
    if 'val_accuracy' in h:
        axes[1].plot(h['val_accuracy'], label='val_acc')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].set_title('准确率曲线')
    
    plt.tight_layout()
    plt.show()

# 绘制前面训练的历史
plot_history(history_v)

### 第二阶段小结

| 步骤 | API | 作用 |
|------|-----|------|
| 编译 | `compile(optimizer, loss, metrics)` | 配置训练规则 |
| 训练 | `fit(x, y, epochs, batch_size, validation_data)` | 训练模型 |
| 评估 | `evaluate(x, y)` | 评估性能 |
| 预测 | `predict(x)` | 预测新数据 |

---
# 第三阶段：工程实践

本阶段学习模型保存、加载，以及训练过程中的回调机制。

---
## §11 模型保存与加载

训练好的模型可以保存到磁盘，之后加载使用。

In [ ]:
# 保存模型
model_v.save('demo_model.keras')
print("模型已保存到 demo_model.keras")

# 加载模型
loaded = keras.models.load_model('demo_model.keras')
print("模型加载成功")

# 验证加载的模型
pred_original = model_v.predict(val_x[:5], verbose=0)
pred_loaded = loaded.predict(val_x[:5], verbose=0)
print(f"预测结果一致: {np.allclose(pred_original, pred_loaded)}")

# 清理临时文件
import os
os.remove('demo_model.keras')

---
## §12 Callbacks：训练过程中的回调

Callback 可以在训练过程中执行自定义操作，例如：

| Callback | 作用 |
|----------|------|
| `EarlyStopping` | 验证指标不再改善时提前停止 |
| `ReduceLROnPlateau` | 验证指标不再改善时降低学习率 |
| `ModelCheckpoint` | 自动保存最佳模型 |
| `TensorBoard` | 记录训练日志供可视化 |

In [ ]:
# 构建模型
model_cb = keras.Sequential([
    layers.Dense(32, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])
model_cb.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 定义回调
callbacks = [
    # 早停：验证损失连续 3 轮不改善就停训
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True  # 恢复最优权重
    ),
    
    # 学习率衰减：验证损失连续 2 轮不改善，学习率乘 0.5
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2
    )
]

# 训练（最多 50 轮，但早停会提前结束）
history_cb = model_cb.fit(
    train_x, train_y,
    epochs=50,
    batch_size=32,
    validation_data=(val_x, val_y),
    callbacks=callbacks,
    verbose=1
)

print(f"\n实际训练了 {len(history_cb.history['loss'])} 轮（早停生效）")

---
# 总结

## Keras 核心工作流

```
1. 定义层 / 搭建模型
   - Sequential：简单的线性堆叠
   - 函数式 API：复杂结构（残差、多输入输出）

2. compile(optimizer, loss, metrics)
   - 配置优化器、损失函数、监控指标

3. fit(x, y, epochs, batch_size, validation_data, callbacks)
   - 训练模型

4. evaluate() / predict()
   - 评估和预测

5. save() / load_model()
   - 保存和加载模型
```

## 损失函数速查表

| 任务 | 标签格式 | 损失函数 | 输出激活 |
|------|---------|---------|---------|
| 二分类 | 0/1 | `BinaryCrossentropy` | sigmoid |
| 多分类 | 整数 | `SparseCategoricalCrossentropy` | softmax |
| 多分类 | one-hot | `CategoricalCrossentropy` | softmax |
| 回归 | 连续值 | `MSE` | 无 |

---
**下一步学习**：
- 用真实数据集（MNIST、CIFAR-10）替换合成数据
- 学习卷积神经网络（CNN）处理图像
- 学习循环神经网络（RNN/LSTM）处理序列